# Day 22 — The Anthropic SDK, in depth

Every previous "API" cell hid the details behind a helper. Today we open it up: the exact
shape of a request, the `Message` response object, the parameters that actually matter, prompt
caching, token counting, multi-turn state, and error handling.

We build a **faithful mock client** so the object model is runnable offline; the real code is
shown alongside and is a drop-in.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | One endpoint: `POST /v1/messages` | 3 min |
| 1 | The request: model, max_tokens, system, messages | 12 min |
| 2 | The response: content blocks, stop_reason, usage | 12 min |
| 3 | Multi-turn: the API is stateless | 8 min |
| 4 | Parameters that matter (and ones that were removed) | 10 min |
| 5 | Prompt caching + token counting + errors | 12 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, time, re
from dataclasses import dataclass, field
print("ready")

ready


## 0 — One endpoint (3 min)

Everything — plain chat, vision, tool use, structured output, extended thinking — is
`POST https://api.anthropic.com/v1/messages`. Tools and output constraints are *fields on this
one request*, not separate APIs. Supporting endpoints (Batches, Files, Token Counting, Models)
feed into it.

```python
import anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY, or an `ant auth login` profile
```

## 1 — The request (12 min)

```python
resp = client.messages.create(
    model="claude-opus-5",              # exact ID string, no date suffix
    max_tokens=1024,                    # REQUIRED. hard ceiling on output tokens
    system="You are a terse assistant.",   # string OR list of text blocks (for caching)
    messages=[
        {"role": "user",      "content": "What's the capital of France?"},
        {"role": "assistant", "content": "Paris."},
        {"role": "user",      "content": "And of Japan?"},
    ],
)
```

### `messages` rules

- alternating-ish `user` / `assistant`; first message must be `user`
- consecutive same-role messages are allowed (API merges them)
- `content` is a **string** or a **list of content blocks** (`text`, `image`, `document`,
  `tool_result`, …)
- the API is **stateless** — you resend the whole history every call (§3)
- **no assistant prefill** on current models (a trailing `assistant` message to steer format
  returns a 400) — use the system prompt or structured outputs instead

In [2]:
# A faithful mock of client.messages.create — same call signature, same response shape.
@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    cache_creation_input_tokens: int = 0
    cache_read_input_tokens: int = 0

@dataclass
class TextBlock:
    text: str
    type: str = "text"

@dataclass
class ToolUseBlock:
    id: str
    name: str
    input: dict
    type: str = "tool_use"

@dataclass
class Message:
    id: str
    model: str
    role: str = "assistant"
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"          # end_turn | max_tokens | tool_use | stop_sequence | refusal
    stop_sequence: str = None
    usage: Usage = field(default_factory=Usage)
    type: str = "message"
    _request_id: str = "req_mock123"

def _count(s): return max(1, len(re.findall(r"\S+", str(s))) * 4 // 3)   # rough; real: count_tokens

class MockMessages:
    def __init__(self, cache={}):
        self._cache = cache
    def create(self, *, model, max_tokens, messages, system=None, stop_sequences=None,
               tools=None, temperature=None, metadata=None, **kw):
        # token accounting
        sys_txt = system if isinstance(system, str) else " ".join(b["text"] for b in (system or []))
        hist = json.dumps(messages)
        in_tok = _count(sys_txt) + _count(hist)
        # cache read: if a system list block was marked cache_control and we've "seen" it
        cread = 0
        if isinstance(system, list):
            key = sys_txt[:50]
            if key in self._cache: cread = self._cache[key]
            else: self._cache[key] = _count(sys_txt)
        # generate a canned reply
        last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
        reply = f"[mock answer to: {str(last_user)[:60]}]"
        out_tok = _count(reply)
        stop = "max_tokens" if out_tok > max_tokens else "end_turn"
        text = reply[: max_tokens * 4] if stop == "max_tokens" else reply
        return Message(id="msg_" + hex(abs(hash(hist)))[2:10], model=model,
                       content=[TextBlock(text=text)], stop_reason=stop,
                       usage=Usage(input_tokens=in_tok - cread, output_tokens=out_tok,
                                   cache_read_input_tokens=cread))

class MockAnthropic:
    def __init__(self): self.messages = MockMessages()

client = MockAnthropic()
resp = client.messages.create(
    model="claude-opus-5", max_tokens=1024,
    system="You are a terse assistant.",
    messages=[{"role": "user", "content": "capital of France?"},
              {"role": "assistant", "content": "Paris."},
              {"role": "user", "content": "and Japan?"}])
print(resp)

Message(id='msg_1dc674ef', model='claude-opus-5', role='assistant', content=[TextBlock(text='[mock answer to: and Japan?]', type='text')], stop_reason='end_turn', stop_sequence=None, usage=Usage(input_tokens=26, output_tokens=6, cache_creation_input_tokens=0, cache_read_input_tokens=0), type='message', _request_id='req_mock123')


## 2 — The response `Message` object (12 min)

| Field | What it is |
| ----- | ---------- |
| `content` | **list of blocks** — `TextBlock`, `ThinkingBlock`, `ToolUseBlock`. Always iterate and check `.type`; `resp.content[0].text` only works if block 0 is text |
| `stop_reason` | `end_turn` (done), `max_tokens` (truncated!), `tool_use` (run tools & loop), `stop_sequence`, `refusal` |
| `usage` | `input_tokens`, `output_tokens`, `cache_creation_input_tokens`, `cache_read_input_tokens` |
| `id` / `model` | echo of what served the request |
| `_request_id` | the `request-id` header — log it when reporting failures (public despite the underscore) |
| `stop_details` | populated **only** when `stop_reason == "refusal"` — `.category`, `.explanation`; `None` otherwise, so guard before reading |

In [3]:
def extract_text(msg):
    return "".join(b.text for b in msg.content if b.type == "text")

def handle(msg):
    if msg.stop_reason == "refusal":
        return f"REFUSED ({getattr(msg, 'stop_details', None)})"
    if msg.stop_reason == "max_tokens":
        return f"TRUNCATED at max_tokens — retry with a higher limit. partial: {extract_text(msg)!r}"
    if msg.stop_reason == "tool_use":
        return "wants tools: " + ", ".join(b.name for b in msg.content if b.type == "tool_use")
    return extract_text(msg)

r1 = client.messages.create(model="claude-opus-5", max_tokens=1024,
                            messages=[{"role": "user", "content": "hello"}])
r2 = client.messages.create(model="claude-opus-5", max_tokens=2,   # tiny -> truncation
                            messages=[{"role": "user", "content": "tell me a long story"}])
print("normal   :", handle(r1), "| usage:", r1.usage)
print("truncated:", handle(r2))

normal   : [mock answer to: hello] | usage: Usage(input_tokens=6, output_tokens=5, cache_creation_input_tokens=0, cache_read_input_tokens=0)
truncated: TRUNCATED at max_tokens — retry with a higher limit. partial: '[mock an'


**`max_tokens` is required and is a hard cliff.** Hitting it truncates mid-sentence and you
pay for a useless partial + a retry. Defaults to reach for: `~16000` for non-streaming (keeps
you under SDK HTTP timeouts), `~64000` for streaming, `~256` for classification, `0` only for
cache pre-warming.

## 3 — The API is stateless (8 min)

There is no server-side conversation. Every call you send the **entire** message list. A
"chat" is a list you keep appending to.

In [4]:
class Conversation:
    def __init__(self, client, model="claude-opus-5", system=None, max_tokens=1024):
        self.client, self.model, self.system, self.max_tokens = client, model, system, max_tokens
        self.messages = []
    def send(self, user_text):
        self.messages.append({"role": "user", "content": user_text})
        resp = self.client.messages.create(model=self.model, max_tokens=self.max_tokens,
                                           system=self.system, messages=self.messages)
        reply = "".join(b.text for b in resp.content if b.type == "text")
        self.messages.append({"role": "assistant", "content": reply})   # <- must append the reply
        return reply, resp.usage

chat = Conversation(client, system="You are helpful.")
chat.send("My name is Sam.")
reply, usage = chat.send("What's my name?")
print(reply)
print(f"turn 2 sent {usage.input_tokens} input tokens — the WHOLE history, growing every turn")
print("messages list now has", len(chat.messages), "entries")

[mock answer to: What's my name?]
turn 2 sent 34 input tokens — the WHOLE history, growing every turn
messages list now has 4 entries


Consequences: input tokens grow every turn (Day 04); long chats need **compaction** (server-
side summary, beta) or client-side truncation; and if you forget to append the assistant reply
the model loses its own last turn.

## 4 — Parameters that matter (10 min)

| Parameter | Notes |
| --------- | ----- |
| `model` | exact string from the table; **never** append a date suffix |
| `max_tokens` | required; hard output ceiling (§2) |
| `system` | persona / rules / format; string, or list of text blocks for `cache_control` |
| `stop_sequences` | up to 4 strings; generation stops before emitting one (`stop_reason="stop_sequence"`) |
| `thinking` | `{"type": "adaptive"}` on current models; **`budget_tokens` removed** (400) on Fable 5 / Opus 5 / 4.7 / 4.8 / Sonnet 5 |
| `output_config` | `{"effort": "low".."max"}` (default `high`); and `{"format": {...}}` for structured output (Day 23) |
| `metadata` | `{"user_id": "..."}` for abuse monitoring — **no PII** |
| `temperature` / `top_p` / `top_k` | **removed / rejected (400)** on the 4.6+ family — don't send them |

Extended thinking + effort replace the old "temperature" and "budget_tokens" knobs. Ask for
"more careful" via `effort`, not sampling params.

In [5]:
# stop_sequences demo (mock honours a simple contains-check)
class MockMessages2(MockMessages):
    def create(self, **kw):
        m = super().create(**kw)
        for s in (kw.get("stop_sequences") or []):
            if s in m.content[0].text:
                m.content[0].text = m.content[0].text.split(s)[0]
                m.stop_reason, m.stop_sequence = "stop_sequence", s
        return m
client.messages = MockMessages2()

r = client.messages.create(model="claude-opus-5", max_tokens=100, stop_sequences=["]"],
                           messages=[{"role": "user", "content": "say something"}])
print("text:", repr(r.content[0].text), "| stop_reason:", r.stop_reason, "| seq:", r.stop_sequence)

# what a removed param does on a real current model:
print("\nOn claude-opus-5:  client.messages.create(..., temperature=0.7)  ->  400 BadRequestError")
print("On claude-opus-5:  thinking={'type':'enabled','budget_tokens':2000}  ->  400 BadRequestError")
print("Use instead:        thinking={'type':'adaptive'}, output_config={'effort':'high'}")

text: '[mock answer to: say something' | stop_reason: stop_sequence | seq: ]

On claude-opus-5:  client.messages.create(..., temperature=0.7)  ->  400 BadRequestError
On claude-opus-5:  thinking={'type':'enabled','budget_tokens':2000}  ->  400 BadRequestError
Use instead:        thinking={'type':'adaptive'}, output_config={'effort':'high'}


## 5 — Caching, token counting, errors (12 min)

### Prompt caching — cache a stable prefix, save up to 90% on it

```python
resp = client.messages.create(
    model="claude-opus-5", max_tokens=1024,
    system=[{"type": "text", "text": BIG_STABLE_INSTRUCTIONS,
             "cache_control": {"type": "ephemeral"}}],       # cache everything up to here
    messages=[{"role": "user", "content": todays_question}], # volatile part AFTER the breakpoint
)
```

Caching is a **prefix match**: any byte change anywhere before the breakpoint invalidates the
whole cache. Render order is `tools` → `system` → `messages`, so put the frozen content first.
Verify with `usage.cache_read_input_tokens` — if it's 0 across identical-prefix calls, a silent
invalidator (a timestamp, a UUID, unsorted JSON, a varying tool list) is in the prefix.

In [6]:
client.messages = MockMessages(cache={})
BIG = "You are a support agent. " + "Follow these 40 rules carefully. " * 40   # a big stable prefix

def ask(q):
    r = client.messages.create(model="claude-opus-5", max_tokens=200,
        system=[{"type": "text", "text": BIG, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": q}])
    return r.usage

u1 = ask("first question")
u2 = ask("second question")   # same prefix -> should be a cache READ
print(f"call 1: input {u1.input_tokens:4d}  cache_read {u1.cache_read_input_tokens}")
print(f"call 2: input {u2.input_tokens:4d}  cache_read {u2.cache_read_input_tokens}  <- prefix served from cache")

call 1: input  279  cache_read 0
call 2: input    6  cache_read 273  <- prefix served from cache


### Token counting — use the API, never `tiktoken`

```python
ct = client.messages.count_tokens(model="claude-opus-5", system=system, messages=messages, tools=tools)
print(ct.input_tokens)     # exact, includes tool schemas and formatting overhead
```

`tiktoken` is OpenAI's tokenizer and gives the wrong number for Claude (the Opus 4.7+
tokenizer differs again). Budget with `count_tokens`.

### Error handling — catch a chain, most-specific first

In [7]:
# sketch of the real handler (exception classes from the anthropic SDK)
HANDLER = '''
import anthropic
try:
    resp = client.messages.create(...)
except anthropic.NotFoundError:            # bad model id / endpoint  -> fix, don't retry
    ...
except anthropic.BadRequestError as e:     # 400 (bad params, prefill, removed knob) -> fix
    ...
except anthropic.AuthenticationError:      # 401 -> bad key
    ...
except anthropic.RateLimitError as e:      # 429 -> back off (SDK already retries 2x)
    retry_after = int(e.response.headers.get("retry-after", "60"))
except anthropic.APIStatusError as e:      # other 4xx/5xx
    if e.status_code >= 500: ...            # retryable
except anthropic.APIConnectionError:       # network
    ...
'''
print(HANDLER)
print("The SDK auto-retries 408/409/429/>=500 + connection errors with backoff (max_retries=2 default).")
print("A single `except Exception` loses the retryable/non-retryable distinction — don't.")


import anthropic
try:
    resp = client.messages.create(...)
except anthropic.NotFoundError:            # bad model id / endpoint  -> fix, don't retry
    ...
except anthropic.BadRequestError as e:     # 400 (bad params, prefill, removed knob) -> fix
    ...
except anthropic.AuthenticationError:      # 401 -> bad key
    ...
except anthropic.RateLimitError as e:      # 429 -> back off (SDK already retries 2x)
    retry_after = int(e.response.headers.get("retry-after", "60"))
except anthropic.APIStatusError as e:      # other 4xx/5xx
    if e.status_code >= 500: ...            # retryable
except anthropic.APIConnectionError:       # network
    ...

The SDK auto-retries 408/409/429/>=500 + connection errors with backoff (max_retries=2 default).
A single `except Exception` loses the retryable/non-retryable distinction — don't.


## 6 — Exercises

1. **Truncation detector.** Wrap `create` so that on `stop_reason == "max_tokens"` it
   automatically retries once with `max_tokens * 2` and concatenates. What could go wrong with
   naive concatenation?
2. **History budget.** Add a `_trim()` to `Conversation` that drops the oldest user/assistant
   *pairs* (never the system) once the estimated input exceeds a budget. Keep the most recent N.
3. **Cache-hit audit.** Add a `datetime.now().isoformat()` string into `BIG` and re-run `ask`
   twice. Show `cache_read_input_tokens` goes to 0, then fix it by moving the timestamp into
   the user message.
4. **Content-block walker.** Write `render(msg)` that turns a mixed `content` list
   (text + tool_use blocks) into a readable string with `[tool: name(args)]` markers.
5. **count_tokens vs len.** For three prompts of increasing tool-schema size, compare
   `_count()` (our rough estimate) to `len(text.split())`. Argue why you still can't trust
   either for billing.
6. **Stateless proof.** Send two independent `create` calls with the *same* messages list.
   Show the responses are independent (no memory between calls) — the only state is what you
   pass in.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. How many API endpoints do plain chat, vision, tool use, and structured output use?
2. What are the two allowed shapes of a message's `content`?
3. Why must you iterate `resp.content` instead of reading `resp.content[0].text`?
4. What does `stop_reason == "max_tokens"` mean and what does it cost you?
5. The API is stateless — what does that require you to do on every turn, and what's the
   consequence for long conversations?
6. Name three request parameters that are rejected with a 400 on current models, and what
   replaced them.
7. Your `cache_read_input_tokens` is 0 across identical-looking calls. What's the likely cause
   and the fix?

## Where this goes next

- **Day 23 — Function calling & structured outputs:** the request/response cycle for tools in
  the API (you built the *loop* on Day 19 — now the *wire format*), plus schema-constrained
  JSON output.